<div style="
    background-color: #7BAFD4;
    border-radius: 18px;
    padding: 22px 26px;
    display: flex;
    align-items: center;
    gap: 24px;
    margin-bottom: 18px;
">
    <img src="592_avatar_lr.png" alt="GEOG 592"
        style="width:110px;height:110px;border-radius:22px;">
    <div>
        <div style="
            font-family: Impact, 'Arial Black', Arial, sans-serif;
            font-size: 42px;
            font-weight: 900;
            letter-spacing: 2px;
            line-height: 1;
            color: white;
        ">GEOG 592</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 22px;
            font-weight: 700;
            color: #13294B;
            margin-top: 8px;
        ">GIS Programming</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 16px;
            color: #13294B;
            margin-top: 6px;
        ">Module 3, Day 2: Cleaning, Validation, and Summaries</div>
    </div>
</div>

# Module 3: Day 2
## Cleaning, Validating, and Summarizing Data with pandas

Today we use pandas to turn the Açaí CSV into a dataset that is safer to analyze.

### Today you will learn to:
- convert numeric-looking strings into numeric data
- detect missing values
- create a Boolean data-quality flag
- distinguish `dropna()` from `fillna()`
- inspect unique and duplicate values
- calculate descriptive summaries
- sort and filter using summary information
- export a cleaned CSV

## Import pandas and load the data

Use `read_csv()` to load the data. The `skiprows` parameter allows you to tell Panda where your data actually starts.

It's good to inspect your data with `head()` after it is loaded.

In [2]:
import pandas as pd
filename = "tabela1613.csv"
df = pd.read_csv(filename, skiprows = 4) 
print(df.columns)
df.head(3)

Index(['Nível', 'Cód.', 'Município', 'Total', 'Unnamed: 4', 'Açaí', 'Units'], dtype='str')


,Nível,Cód.,Município,Total,Unnamed: 4,Açaí,Units
0,MU,1100015,Alta Floresta D'Oeste (RO),..,Quilogramas por Hectare,6000,Quilogramas por Hectare
1,MU,1100023,Ariquemes (RO),..,Quilogramas por Hectare,9167,Quilogramas por Hectare
2,MU,1100031,Cabixi (RO),..,Quilogramas por Hectare,-,Quilogramas por Hectare


Let's translate the column names in Pandas (you could also have done this in Excel before loading the file).

use a `print()` statement or `head()` to verify the new column names.


In [3]:
df.columns = ['MU', 'geoID', 'name', 'empty', 'units', 'acai', 'units2']
print(df['geoID'].iloc[1])
df.head(5)

1100023


,MU,geoID,name,empty,units,acai,units2
0,MU,1100015,Alta Floresta D'Oeste (RO),..,Quilogramas por Hectare,6000,Quilogramas por Hectare
1,MU,1100023,Ariquemes (RO),..,Quilogramas por Hectare,9167,Quilogramas por Hectare
2,MU,1100031,Cabixi (RO),..,Quilogramas por Hectare,-,Quilogramas por Hectare
3,MU,1100049,Cacoal (RO),..,Quilogramas por Hectare,-,Quilogramas por Hectare
4,MU,1100056,Cerejeiras (RO),..,Quilogramas por Hectare,-,Quilogramas por Hectare


More cleaning: Let's remove the unnecesary columns. 

In [4]:
df2 = df[["geoID", "name", "acai", "units"]].copy()
df2.head()

,geoID,name,acai,units
0,1100015,Alta Floresta D'Oeste (RO),6000,Quilogramas por Hectare
1,1100023,Ariquemes (RO),9167,Quilogramas por Hectare
2,1100031,Cabixi (RO),-,Quilogramas por Hectare
3,1100049,Cacoal (RO),-,Quilogramas por Hectare
4,1100056,Cerejeiras (RO),-,Quilogramas por Hectare


### Why `.copy()`?

The expression `df[["geoID", "name", "acai", "units"]]` returns a filtered DataFrame.

> ```python
> df_filtered = df[["geoID", "name", "acai", "units"]]
> ```
> This example function call creates a variable `df_filtered` that can actually refer to > the same memory as `df`, so *modifying `df2` would also modify your original dataset as a side effect, and possibly produce a warning.

Instead, we use `.copy()` to get an **independent** working object that we can modify without warnings or side-effects.

## Data types
Now let's check the datatypes in the file.

In [5]:
df2.dtypes

geoID    str
name     str
acai     str
units    str
dtype: object

Several columns still look numeric but are stored as strings. A CSV file does not guarantee that values are interpreted with the type we want.

## Convert `Data_Value` to Numeric

`pd.to_numeric()` converts a Series to numbers.

Using `errors="coerce"` means that values that cannot be converted become `NaN` rather than stopping the program.

In [5]:
df2["acai"] = pd.to_numeric(
    df2["acai"],
    errors="coerce"
)

# df2["acai"].dtype
df2["acai"].head()

0    6000.0
1    9167.0
2       NaN
3       NaN
4       NaN
Name: acai, dtype: float64

### What is `NaN`?

`NaN` is an abbreviation for **Not a Number**. pandas commonly uses it to represent missing numeric data.

Detect Missing Values

`isna()` returns `True` where data are missing.

In [7]:
df2["acai"].isna().head(10)

0    False
1    False
2     True
3     True
4     True
5     True
6     True
7    False
8     True
9    False
Name: acai, dtype: bool

Count the missing values by adding the Boolean values. In Python, `True` behaves like 1 and `False` like 0 in this context.

In [8]:
missing_count = df2["acai"].isna().sum()
print("Missing Data_Value records:", missing_count)

Missing Data_Value records: 5259


### Missing vs. misinterpreted

We treated all non-numeric data as "missing," but actually there is metadata at the end of the csv explaining the meaning of some values that can't be interpreted as numbers. 
(Commonly, metadata are placed at the top, or in another table, but in this CSV it's at the bottom of the dataset). 


In [9]:
# Since we made a copy, we have not changed the original data frame 
# so we can read the metadata

df.tail(8)

,MU,geoID,name,empty,units,acai,units2
5556,Legenda,NaN,NaN,NaN,NaN,NaN,NaN
5557,Símbolo,Significado,NaN,NaN,NaN,NaN,NaN
5558,-,"Zero absoluto, não resultante de um cálculo ou...",NaN,NaN,NaN,NaN,NaN
5559,0,Zero resultante de um cálculo ou arredondament...,NaN,NaN,NaN,NaN,NaN
5560,X,Valor inibido para não identificar o informant...,NaN,NaN,NaN,NaN,NaN
5561,..,Valor não se aplica.\r\nEx: Não se pode obter ...,NaN,NaN,NaN,NaN,NaN
5562,...,Valor não disponível.\r\nEx: A produção de fei...,NaN,NaN,NaN,NaN,NaN
5563,A a Z\r\n(exceto X),Significa uma faixa de valores. Varia em funçã...,NaN,NaN,NaN,NaN,NaN


From the data description we learn that '-' means "Absolute zero, not resulting from a calculation or rounding. Example: In a given municipality, there are no 14-year-old people with no formal education."
and that  the `'...'` means  "Value unavailable. Example: Bean production in a particular municipality was not surveyed, or the municipality did not exist in the year of the survey."

So depending on how we want to display information `'-'` could be 0 and `'...'` could be `NaN`. For the moment we are going to treat them both as `NaN`. 

### Missingness as a data-quality flag

Sometimes it is useful to create a new Boolean column that records whether a value is missing. This is often called a **missing-data indicator** or a **data-quality flag**.

In [10]:
df2["Data_Value_missing"] = df2["acai"].isna()
df2.head(10)

,geoID,name,acai,units,Data_Value_missing
0,1100015,Alta Floresta D'Oeste (RO),6000.0,Quilogramas por Hectare,False
1,1100023,Ariquemes (RO),9167.0,Quilogramas por Hectare,False
2,1100031,Cabixi (RO),NaN,Quilogramas por Hectare,True
3,1100049,Cacoal (RO),NaN,Quilogramas por Hectare,True
4,1100056,Cerejeiras (RO),NaN,Quilogramas por Hectare,True
5,1100064,Colorado do Oeste (RO),NaN,Quilogramas por Hectare,True
6,1100072,Corumbiara (RO),NaN,Quilogramas por Hectare,True
7,1100080,Costa Marques (RO),6800.0,Quilogramas por Hectare,False
8,1100098,Espigão D'Oeste (RO),NaN,Quilogramas por Hectare,True
9,1100106,Guajará-Mirim (RO),3833.0,Quilogramas por Hectare,False


This preserves information about *which rows* had a problem, rather than only reporting a total count. 

### `dropna()` vs. `fillna()`

There are different ways to handle missing values.

1.  Remove records with missing values

    ```python
    df2.dropna(subset=["acai"])
    ```
2. Replace missing values
    ```python
    df2["acai"].fillna(...)
    ```

Do **not** replace missing values without thinking about what your choice means!
Filling with 0, a mean, or another value changes the data and should be justifiable.

In [11]:
analysis = df2.dropna(subset=["acai"]).copy()
print("Rows available for analysis:", len(analysis))

Rows available for analysis: 305


## Unique Values

`unique()` shows the distinct values. `nunique()` counts them.

In [6]:
## check if all the values are stored in the same units
df2["units"].nunique()


1

In [7]:
print(len(df), 'in the original df')
print(len(df2[df2["units"] == 'Quilogramas por Hectare']), 'number of rows with Quilogramas por Hectare')
print('difference:',len(df) - len(df2[df2["units"] == 'Quilogramas por Hectare']) )

5564 in the original df
5541 number of rows with Quilogramas por Hectare
difference: 23


In [14]:
## checking if we have repeating location values 
df2["geoID"].nunique() 
print('number of repeating values:', len(df2["geoID"]) - df2["geoID"].nunique() ) 

number of repeating values: 16


In [15]:
# Note that the metadata is causing issues
df2["geoID"].tail()

5559    Zero resultante de um cálculo ou arredondament...
5560    Valor inibido para não identificar o informant...
5561    Valor não se aplica.\r\nEx: Não se pode obter ...
5562    Valor não disponível.\r\nEx: A produção de fei...
5563    Significa uma faixa de valores. Varia em funçã...
Name: geoID, dtype: str

Obviously there is some information in that table that we do not need. Let's find a way to solve this (without using Excel).

In [16]:
df2["geoID"] = pd.to_numeric(
    df2["geoID"],
    errors="coerce"
).astype("Int64")
missing_count = df2["geoID"].isna().sum()
print('number of NA:', missing_count)

number of NA: 23


I am getting the idea that we have 22 rows of non-numeric information, but before we delete them I want to show you an example of how `value_counts()` can be used, since it is a very useful method. 
First I am going to create two new columns that will be constructed from the `name` column by extracting the values inside the parenthesis. So for example; Cabixi (RO) will be seperated into Cabixi and RO. 

In [17]:
df2[["Municipality", "State"]] = df2["name"].str.extract(r"^(.*?)\s*\(([^)]+)\)$") ## I am using regex, and as mentioned in class, I like using AI to create regex these days. 
df2.head()

,geoID,name,acai,units,Data_Value_missing,Municipality,State
0,1100015,Alta Floresta D'Oeste (RO),6000.0,Quilogramas por Hectare,False,Alta Floresta D'Oeste,RO
1,1100023,Ariquemes (RO),9167.0,Quilogramas por Hectare,False,Ariquemes,RO
2,1100031,Cabixi (RO),NaN,Quilogramas por Hectare,True,Cabixi,RO
3,1100049,Cacoal (RO),NaN,Quilogramas por Hectare,True,Cacoal,RO
4,1100056,Cerejeiras (RO),NaN,Quilogramas por Hectare,True,Cerejeiras,RO


In [18]:
## now let's count the number of how many instances we have of each state
df2["State"].value_counts()

State
MG    852
SP    638
RS    494
BA    417
PR    399
SC    294
GO    243
PB    223
PI    222
MA    217
CE    184
PE    184
RN    166
PA    143
MT    140
TO    139
AL    101
RJ     89
MS     79
ES     78
SE     71
AM     62
RO     52
AC     22
AP     16
RR     15
DF      1
Name: count, dtype: int64

From Wikipedia, The Federal Dristict (DF) is Located in the Center-West Region, it is the smallest Brazilian federal unit and the only one that has no municipalities. Also from Wikipedia: Minas Gerais' territory is subdivided into 853 municipalities, the largest number among Brazilian states. 

## Check for Duplicate IDs

A geographic identifier such as `LocationID` should generally identify one county record in this dataset.

In [19]:
duplicate_ids = df2["geoID"].duplicated().sum()
print("Number of duplicate geoID:", duplicate_ids)

Number of duplicate geoID: 22


To see the actual duplicate rows, use the Boolean result from `duplicated()` as a filter.

In [20]:
df2[df2["geoID"].duplicated(keep=False)][
    ["name", "State", "geoID"]]

,name,State,geoID
5541,NaN,NaN,<NA>
5542,NaN,NaN,<NA>
5543,NaN,NaN,<NA>
5544,NaN,NaN,<NA>
5545,NaN,NaN,<NA>
5546,NaN,NaN,<NA>
5547,NaN,NaN,<NA>
5548,NaN,NaN,<NA>
5549,NaN,NaN,<NA>
5550,NaN,NaN,<NA>


Let's go ahead and delete rows with NaN in several columns using dropNA.  

In [21]:
df2.dropna(
    subset=["name", "State", "geoID", "acai"], ## this expects the NaN to be in all these columns
    how="all", ## row is removed only when every specified column is missing.
    inplace=True ## modify the existing df2 directly.
)

## check for duplicates once it has been cleaned of NaN NaN NaN
df2[df2["geoID"].duplicated(keep=False)][
    ["name", "State", "geoID", "acai"]]

,name,State,geoID,acai


In [22]:
## Notice that we have changed the shape of df2
print(df2.shape)
print(df.shape)

(5541, 7)
(5564, 7)


## Descriptive Statistics

Once `Acai` is numeric, we can calculate summaries directly from the Series object.

In [23]:
print("Minimum:", df2["acai"].min())
print("Maximum:", df2["acai"].max())
print("Median:", df2["acai"].median())
print("Mean:", df2["acai"].mean())



Minimum: 200.0
Maximum: 43200.0
Median: 6333.0
Mean: 7043.43606557377


### Why compare mean and median?

The mean and median describe the center of a distribution in different ways. Their relationship can give us an early clue about whether a distribution is symmetric or skewed.

Tomorrow we will look at the distribution visually.

In [24]:
df2["acai"].describe()

count      305.000000
mean      7043.436066
std       4216.206671
min        200.000000
25%       4000.000000
50%       6333.000000
75%      10000.000000
max      43200.000000
Name: acai, dtype: float64

## Sort the Data

Sort by acai production value (highest to lowest).

In [25]:
df2.sort_values(
    "acai",
    ascending=False
)[["name", "State", "acai"]].head(10)

,name,State,acai
1020,Paracuru (CE),CE,43200.0
989,Limoeiro do Norte (CE),CE,21250.0
1061,Tianguá (CE),CE,15400.0
1064,Ubajara (CE),CE,15333.0
220,Medicilândia (PA),PA,15000.0
960,Ibiapina (CE),CE,15000.0
1071,Viçosa do Ceará (CE),CE,15000.0
106,Juruá (AM),AM,14211.0
302,Macapá (AP),AP,13654.0
124,Santo Antônio do Içá (AM),AM,13514.0


### Which municipio has the maximum value?

Instead of scanning the table manually, combine sorting with `iloc`.

In [26]:
highest = df2.sort_values(
    "acai",
    ascending=False
).iloc[0]

print(highest["name"], highest["acai"])

Paracuru (CE) 43200.0


or use the max value in case you have two counties with similar max values (not the case in this example)

In [27]:
max_Data_Value = df2['acai'].max()
print(df2['name'][df2['acai']== max_Data_Value] )

1020    Paracuru (CE)
Name: name, dtype: str


### A First Look at `groupby()`

`groupby()` lets us divide rows into groups and calculate a summary for each group.

We can ask for the average `acai` in each state:

In [28]:
results = df2.groupby("State")["acai"].sum()
print(results.sort_values(ascending=False))

State
PA    776831.0
AM    573839.0
RO    190164.0
CE    165183.0
AP    120295.0
MA     93786.0
BA     82321.0
RR     57238.0
TO     39297.0
ES     18667.0
AL     11538.0
AC      6367.0
MT      6000.0
PE      4500.0
RN      2222.0
SP         0.0
SE         0.0
SC         0.0
RS         0.0
DF         0.0
GO         0.0
MG         0.0
PR         0.0
PI         0.0
PB         0.0
MS         0.0
RJ         0.0
Name: acai, dtype: float64


Read the expression from left to right:

1. `df2` is the DataFrame.
2. `.groupby("State")` creates groups based on that column.
3. `["acai"]` selects the numeric variable we want to summarize.
4. `.sum()` calculates the sum for each group.

This is only an introduction to `groupby()`. We will use grouping again later when spatial joins create categories such as points within different polygons.

## Export the Cleaned Data

pandas can write a DataFrame directly to CSV.

In [29]:
df2.to_csv(
    "cleaned_acai.csv",
    index=False
)

The `index=False` argument prevents pandas from writing its row index as an extra CSV column.

## Some questions: 

1. What does `errors="coerce"` do?
    
    Values that cannot be converted (into numeric) are turned into NaN (Not a Number).

2. What is the difference between detecting missing values and deciding what to do about them?

    Detecting missing values, use .isna(), deciding what to do about them is tricky because it can skew your data; use .fillna() to replace missing values or .dropna() to remove missing values. 

3. Why might a missing-data flag be useful?

    To see where there are discontinuties in your data and/or mistakes that need to be fixed. 

4. Why should you avoid automatically replacing every missing value with 0?

    It can incorrectly represent your data, by skewing it. (Changes the data)

5. How does `selected_columns.to_csv()` demonstrate dot notation?

    It selected_columns is the object and .to_csv() is the method acting upon the object. 